In [0]:
from pyspark.sql.functions import col

BRONZE_PATH = "s3://s3-goodreads-data/Bronze"
SILVER_PATH = "s3://s3-goodreads-data/Silver/authors"
AUTHORS_BRONZE_PATH = f"s3://s3-goodreads-data/Bronze/authors"

df_raw = spark.read.format("delta").load(AUTHORS_BRONZE_PATH)

# 1. Optimized Clean & Deduplicate
df_clean = (
    df_raw.filter(col("author_id").isNotNull() & col("name").isNotNull())
          .dropDuplicates(["author_id"])
)

# 2. Write to New Delta Path
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_PATH)

# 3. Optimize Table
spark.sql(f"OPTIMIZE delta.`{SILVER_PATH}` ZORDER BY (author_id)")

In [0]:
from delta.tables import DeltaTable

BRONZE_PATH = "s3://s3-goodreads-data/Bronze"
SILVER_PATH = "s3://s3-goodreads-data/Silver"

# 1. BOOKS
df_books = spark.read.format("delta").load(f"{BRONZE_PATH}/books")

df_books.dropDuplicates(["book_id"]) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{SILVER_PATH}/books")

# Optimize
spark.sql(f"OPTIMIZE delta.`{SILVER_PATH}/books`")
print("✅ Silver Books done.")

# 2. INTERACTIONS
df_interactions = spark.read.format("delta").load(f"{BRONZE_PATH}/interactions")

df_interactions.dropDuplicates(["user_id", "book_id"]) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{SILVER_PATH}/interactions")

# Optimize 
spark.sql(f"OPTIMIZE delta.`{SILVER_PATH}/interactions`")
print("✅ Silver Interactions done.")